# 80 · Streaming — Redpanda, the event-streaming backbone

**Everything so far in this series was *at rest*.** The format notebooks (`01`–`04`) wrote a
file and read it back; the query notebooks (`20`–`22`) ran a `SELECT` over a table that was
already sitting in a store. This notebook reaches the layer *underneath* all of that — the
**event stream**, where records are in **motion**: produced one at a time, appended to a log,
and consumed by whoever is listening, whenever they listen.

That layer is **Redpanda**. It speaks the **Apache Kafka protocol** wire-for-wire — the same
brokers, topics, partitions, offsets and consumer groups — so every Kafka client library talks
to it unchanged. It just happens to be a single C++ binary with no JVM and no ZooKeeper. In this
mesh it is the backbone that feeds the streaming jobs: the RAG streaming indexer, the Debezium
CDC pipelines, and the real-time analytics (RTA) rollups.

### The vocabulary, before we touch it

- **Topic** — a named, append-only log of records (events). Producers append; consumers read.
- **Partition** — a topic is split into partitions for parallelism. **Ordering is guaranteed
  *within* a partition, never across them.** A record's key decides its partition, so all events
  for one key land on one partition in order.
- **Offset** — a record's monotonic position *within its partition*. Consuming is just "read
  from offset N onward". Nothing is deleted when you read it; the log stays put.
- **Consumer group** — one or more consumers sharing a `group.id`. The group's committed offset
  is *where it has read up to*; partitions are divided among the group's members. Two different
  groups read the **same** log **independently** — this is the fan-out that lets the RAG indexer
  and a CDC job both consume without stepping on each other.

### Batch vs stream — the mental model

> **Batch asks "what is the state *now*?" and scans a table. Streaming asks "what
> *happened*?" and replays a log of events, in order, as they arrive.**

A table holds the *current* value of each row; an event log holds *every change*, and the table
is just a fold over that log. That inversion — log first, state derived — is what makes
streaming the natural home for change-data-capture, real-time pipelines, and decoupling
producers from consumers.

### Redpanda = Kafka API + a built-in Schema Registry

Redpanda bundles a **Schema Registry** on the same service (port `8081`). Producers register an
Avro (or Protobuf/JSON) schema there and stamp each message with a tiny **schema id**; consumers
look that id up and decode. The payload therefore carries *no field names* — just the id and the
packed values — which is exactly the compact, evolvable **Avro** wire format notebook `03`
dissected in the abstract. This notebook shows it live: a real registered schema, and real
messages on the wire decoded through it.

> **Read-only, throughout.** We **list** topics, **read** schemas, and **consume** a bounded
> batch of messages with a fresh, disposable consumer group. Nothing is produced; no topic is
> created, altered, or deleted. `AdminClient` is used *only* to list metadata.

## Setup — one Kafka client, env-driven, in-cluster defaults

We install **`confluent-kafka[avro]`** — the same client the mesh's own streaming service
(`services/rag-index/consumer.py`) uses. The plain `confluent-kafka` wheel bundles `librdkafka`,
and the **`[avro]`** extra pulls the Schema-Registry client plus its HTTP and Avro dependencies,
so the registry-backed `AvroDeserializer` works out of the box. `polars` (already in the image)
renders every result frame, exactly as in notebooks `20`–`22`; `requests` hits the Schema
Registry's REST API directly for the schema-inspection section.

Every endpoint is **env-driven**, the same pattern the rest of the series uses. The committed
defaults are the **in-cluster** service DNS names; a validation run overrides them via env
(`REDPANDA_BOOTSTRAP`, `SCHEMA_REGISTRY_URL`) **without editing the notebook**. Redpanda here is
**unmeshed, headless ClusterIP, and has no auth** (plaintext, LAN-only, in-cluster) — so there is
no password to load and nothing secret in any cell.

In [1]:
%pip install -q "confluent-kafka[avro]" polars requests


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import contextlib
import io
import os
import time
import uuid
import warnings

import requests
import polars as pl

# confluent-kafka's schema-registry client pulls in authlib, which force-emits a benign
# deprecation warning about httpx at import time (it bypasses the normal warnings filter).
# Trigger that import ONCE here under a swallowed stderr so the rest of the notebook stays clean.
warnings.filterwarnings("ignore")
with contextlib.redirect_stderr(io.StringIO()):
    import authlib.integrations.httpx_client  # noqa: F401 -- imported only to absorb its import-time warning

# committed defaults = in-cluster DNS; a validation run overrides these via env, no edit needed.
BOOTSTRAP = os.environ.get("REDPANDA_BOOTSTRAP", "redpanda.data-mesh.svc.cluster.local:9092")
SCHEMA_REGISTRY_URL = os.environ.get("SCHEMA_REGISTRY_URL", "http://redpanda.data-mesh.svc.cluster.local:8081")

# small helper: rows + column names -> a polars DataFrame for display (house style, as in 20-22)
def frame(rows, cols):
    return pl.DataFrame(rows, schema=list(cols), orient="row")

print("config resolved — Redpanda broker + Schema Registry endpoints from env (no auth, in-cluster)")

config resolved — Redpanda broker + Schema Registry endpoints from env (no auth, in-cluster)


## 1 · Connect and list topics — `AdminClient`

The `AdminClient` talks the Kafka **admin protocol**: `list_topics()` returns cluster metadata —
every topic and its partitions — which both proves we are connected and shows us what streams
exist. As everywhere in this series, we let the connection prove itself by **what it reports
back** (the broker's topic list), not by echoing the endpoint we dialed.

A live Kafka cluster carries two kinds of topic:

- **Internal / system topics** — bookkeeping the platform keeps for itself:
  `__consumer_offsets` (where every group's committed offsets live), `_schemas` (the Schema
  Registry's own backing log), the `_connect_*` topics (Kafka Connect state), and DataHub's
  `MetadataChangeLog_*` / `*_v1` event topics. By convention these start with `_` / `__` or are
  platform-owned. We flag them so they don't get mistaken for application data.
- **Application topics** — the actual data streams: `rag.chunks` (the RAG streaming indexer),
  `cdc.musicbrainz.public.cdc_demo` (a Debezium CDC feed — the focus of notebook `81`),
  `datasets.*` dataset streams, and `analytics.*` real-time rollups.

In [3]:
from confluent_kafka.admin import AdminClient

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
md = admin.list_topics(timeout=15)          # cluster metadata — proves the connection
print(f"connected — broker cluster reports {len(md.topics)} topics (endpoint from env)")

# classify: internal/system bookkeeping vs application data streams
PLATFORM_PREFIXES = ("_", "MetadataChange", "FailedMetadata", "PlatformEvent", "DataHub")
def is_internal(name):
    return name.startswith(PLATFORM_PREFIXES)

rows = sorted(
    ([t.topic, len(t.partitions), "internal" if is_internal(t.topic) else "app"]
     for t in md.topics.values()),
    key=lambda r: (r[2] != "app", r[0]),      # app topics first, then alphabetical
)
topics_df = frame(rows, ["topic", "partitions", "class"])
app_topics = [r[0] for r in rows if r[2] == "app"]
print(f"{len(app_topics)} application topics, {len(rows) - len(app_topics)} internal/system topics\n")
topics_df

connected — broker cluster reports 21 topics (endpoint from env)
8 application topics, 13 internal/system topics



topic,partitions,class
str,i64,str
"""analytics.health.state_risk""",1,"""app"""
"""analytics.music.artist_tier""",1,"""app"""
"""cdc.musicbrainz.public.cdc_dem…",1,"""app"""
"""datasets.health.big_five""",3,"""app"""
"""datasets.health.brfss""",3,"""app"""
…,…,…
"""__consumer_offsets""",3,"""internal"""
"""_connect_configs""",1,"""internal"""
"""_connect_offsets""",25,"""internal"""


## 2 · The Schema Registry — what a topic's messages *are*

A Confluent-framed Avro message is **not** self-describing the way an Avro *container file* is
(notebook `03`): the payload is a one-byte magic, a **4-byte schema id**, then the packed values.
The schema itself lives in the **Registry**, keyed by **subject**. The near-universal convention
is one subject per topic named `<topic>-value` (and `<topic>-key` when the key is structured too).

We hit the Registry's REST API directly with `requests`:

- `GET /subjects` — every registered subject.
- `GET /subjects/<subject>/versions/latest` — the latest schema for one subject, with its global
  **id** and **version**.

Then we pick a **real, registered** topic (resolved from the subject list — never assumed) and
render its Avro fields. We centre on **`rag.chunks`**, the stream the RAG indexer produces: its
`weyland.rag.RagChunk` record is a clean, real-world Avro schema with a nullable-union on nearly
every field (the evolution-friendly shape notebook `03` explained) and an embedded `array<float>`
carrying a 768-dimension embedding vector.

In [4]:
import json

subjects = requests.get(f"{SCHEMA_REGISTRY_URL}/subjects", timeout=10).json()
print(f"registry holds {len(subjects)} subjects:")
for s in sorted(subjects):
    print("  ", s)

# resolve the value-subject for our centre topic from the live list -- never assume it exists
CENTRE_TOPIC = "rag.chunks"
value_subject = next(s for s in subjects if s == f"{CENTRE_TOPIC}-value")
latest = requests.get(f"{SCHEMA_REGISTRY_URL}/subjects/{value_subject}/versions/latest", timeout=10).json()
avro_schema = json.loads(latest["schema"])
print(f"\nsubject {value_subject}: schema id={latest['id']}, version={latest['version']}")
print(f"record: {avro_schema.get('namespace', '')}.{avro_schema['name']}")

registry holds 7 subjects:
   cdc.musicbrainz.public.cdc_demo-key
   cdc.musicbrainz.public.cdc_demo-value
   datasets.health.big_five-value
   datasets.health.brfss-value
   datasets.health.nhis-value
   datasets.music.lastfm-value
   rag.chunks-value

subject rag.chunks-value: schema id=7, version=1
record: weyland.rag.RagChunk


Render the schema's fields as a frame. Avro has no implicit nullability — a field is nullable
only if `null` is one branch of a **union**, so the `nullable` column below is read straight off
the shape of each field's type. This is the contract every producer and consumer of `rag.chunks`
agrees on, enforced at publish time by the Registry.

In [5]:
def describe_type(t):
    """Flatten an Avro field type into (type_str, nullable)."""
    if isinstance(t, list):                       # a union
        branches = [b for b in t if b != "null"]
        nullable = "null" in t
        inner = ", ".join(describe_type(b)[0] for b in branches)
        return (inner if len(branches) == 1 else f"union[{inner}]"), nullable
    if isinstance(t, dict):                        # array / enum / record / logicalType
        if t.get("type") == "array":
            return f"array<{describe_type(t['items'])[0]}>", False
        return t.get("logicalType") or t.get("type"), False
    return t, False

field_rows = []
for f in avro_schema["fields"]:
    tstr, nullable = describe_type(f["type"])
    field_rows.append([f["name"], tstr, nullable, str(f.get("default", ""))])
frame(field_rows, ["field", "type", "nullable", "default"])

field,type,nullable,default
str,str,bool,str
"""source_path""","""string""",false,""""""
"""op""","""string""",false,""""""
"""chunk_index""","""int""",true,"""None"""
"""chunk_title""","""string""",true,"""None"""
"""chunk_text""","""string""",true,"""None"""
"""vector""","""array<float>""",true,"""None"""
"""content_hash""","""string""",true,"""None"""
"""source_name""","""string""",true,"""None"""
"""kind""","""string""",true,"""None"""


## 3 · Consume a bounded batch — Avro-deserialized off the wire

Now the core move: **read messages that are actually on the topic** and decode them through the
Registry. Three deliberate choices make this safe on a *live* stream:

1. **A fresh, unique consumer group** (`nb80-reader-<uuid>`) with **`enable.auto.commit=False`** —
   we never commit an offset, so this run leaves no mark on any existing group and cannot disturb
   the RAG indexer. `auto.offset.reset="earliest"` means a brand-new group starts at the *head*
   of the log and can therefore see the history that is already there.
2. **A `DeserializingConsumer` with a registry-backed `AvroDeserializer`.** We pass *no* schema
   string: the deserializer reads the 4-byte id off each message and fetches the writer schema
   from the Registry itself — so the same code decodes *any* Confluent-Avro topic.
3. **The poll loop is bounded by time AND count.** We stop at `MAX_MESSAGES` *or* when
   `MAX_SECONDS` elapses — whichever comes first — so on an idle or endless topic the cell always
   returns instead of hanging. This is the single most important habit when reading a live stream
   from a notebook.

We also **resolve the topic against reality**: our centre topic is `rag.chunks`, but if it
happens to be empty we fall through to any other topic that has both a registered `-value` schema
and messages waiting — so the cell shows real decoded data rather than an empty frame.

In [6]:
from confluent_kafka import Consumer, TopicPartition

def message_count(topic):
    """Sum (high - low) watermark across a topic's partitions = messages currently retained."""
    if topic not in md.topics:
        return 0
    probe = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": f"nb80-probe-{uuid.uuid4()}",
                      "enable.auto.commit": False})
    total = 0
    for p in md.topics[topic].partitions:
        lo, hi = probe.get_watermark_offsets(TopicPartition(topic, p), timeout=10)
        total += (hi - lo)
    probe.close()
    return total

# candidate topics = those with a registered -value schema; centre topic first, then the rest.
value_topics = [s[:-len("-value")] for s in subjects if s.endswith("-value")]
candidates = [CENTRE_TOPIC] + [t for t in value_topics if t != CENTRE_TOPIC]
consume_topic = next((t for t in candidates if message_count(t) > 0), None)
if consume_topic is None:
    raise RuntimeError("no schema-registered topic currently has messages to consume")
print(f"consuming from: {consume_topic}  ({message_count(consume_topic)} messages retained)")

consuming from: rag.chunks  (2755 messages retained)


In [7]:
from confluent_kafka import DeserializingConsumer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroDeserializer
from confluent_kafka.serialization import StringDeserializer

MAX_MESSAGES = 10          # bound by COUNT ...
MAX_SECONDS = 15.0         # ... AND by time -- whichever comes first, so this never hangs
GROUP_ID = f"nb80-reader-{uuid.uuid4()}"    # fresh, disposable, read-only group

sr = SchemaRegistryClient({"url": SCHEMA_REGISTRY_URL})
avro_deser = AvroDeserializer(sr)           # no schema string: writer schema fetched by wire id

consumer = DeserializingConsumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": GROUP_ID,
    "key.deserializer": StringDeserializer("utf_8"),
    "value.deserializer": avro_deser,
    "auto.offset.reset": "earliest",        # brand-new group -> start at the head of the log
    "enable.auto.commit": False,            # NEVER commit -> leaves no trace, read-only
})
consumer.subscribe([consume_topic])

records, deadline = [], time.time() + MAX_SECONDS
while len(records) < MAX_MESSAGES and time.time() < deadline:
    msg = consumer.poll(1.0)
    if msg is None:            # no message this second -- keep polling until the deadline
        continue
    if msg.error():            # non-fatal consume error -- log via the stream and move on
        print("consume warning:", msg.error())
        continue
    records.append({"partition": msg.partition(), "offset": msg.offset(), "value": msg.value()})

consumer.close()               # release the group cleanly; nothing was committed
print(f"consumed {len(records)} messages from {consume_topic} "
      f"(bounded at {MAX_MESSAGES} msgs / {MAX_SECONDS:.0f}s)")

consumed 10 messages from rag.chunks (bounded at 10 msgs / 15s)


Render the decoded records. Each value came off the wire as packed Avro bytes and was
reconstructed into a Python dict by the Registry-backed deserializer. The `vector` field is a
768-float embedding array, so we show its **dimension** rather than dumping 768 numbers, and we
truncate the free-text chunk body — the point is that every field decoded, with its correct
type, straight from the stream.

In [8]:
def summarize(v):
    """Turn one decoded record into a compact display row (don't dump 768-float vectors / long text)."""
    out = {}
    for k, val in v.items():
        if isinstance(val, list):
            out[k] = f"<vector dim={len(val)}>"
        elif isinstance(val, str) and len(val) > 48:
            out[k] = val[:45] + "..."
        else:
            out[k] = val
    return out

if records:
    display_rows = []
    for r in records:
        row = {"partition": r["partition"], "offset": r["offset"]}
        row.update(summarize(r["value"]))
        display_rows.append(row)
    cols = list(display_rows[0].keys())
    decoded_df = frame([[row.get(c) for c in cols] for row in display_rows], cols)
    print(f"{len(records)} Avro-decoded records from {consume_topic}:")
    decoded_df
else:
    print("no records decoded")
decoded_df if records else None

10 Avro-decoded records from rag.chunks:


partition,offset,source_path,op,chunk_index,chunk_title,chunk_text,vector,content_hash,source_name,kind,run_id
i64,i64,str,str,i64,null,str,str,str,str,str,str
0,3108,"""nodes/mother/lab/weyland-platf…","""delete""",null,null,null,null,null,null,null,"""554f55a6-c743-4225-b566-e03655…"
0,3109,"""nodes/mother/lab/weyland-platf…","""upsert""",0,null,"""# B135 — LAN NodePort for the …","""<vector dim=768>""","""fcc4843f3290142755ae62c9345743…","""argocd-lan.yaml""","""code""","""554f55a6-c743-4225-b566-e03655…"
0,3110,"""nodes/mother/lab/weyland-platf…","""upsert""",1,null,""" lab is LAN-only and single-op…","""<vector dim=768>""","""fcc4843f3290142755ae62c9345743…","""argocd-lan.yaml""","""code""","""554f55a6-c743-4225-b566-e03655…"
0,3111,"""nodes/mother/lab/weyland-platf…","""upsert""",2,null,""".kubernetes.io/name: argocd-se…","""<vector dim=768>""","""fcc4843f3290142755ae62c9345743…","""argocd-lan.yaml""","""code""","""554f55a6-c743-4225-b566-e03655…"
0,3112,"""nodes/mother/lab/weyland-platf…","""delete""",null,null,null,null,null,null,null,"""554f55a6-c743-4225-b566-e03655…"
0,3113,"""nodes/mother/lab/weyland-platf…","""upsert""",0,null,"""# Feast UI — the registry BROW…","""<vector dim=768>""","""bf9f5407edf559a218732a2f46ae31…","""feast-ui.yaml""","""code""","""554f55a6-c743-4225-b566-e03655…"
0,3114,"""nodes/mother/lab/weyland-platf…","""upsert""",1,null,""" apiVersion: v1 kind: ConfigMa…","""<vector dim=768>""","""bf9f5407edf559a218732a2f46ae31…","""feast-ui.yaml""","""code""","""554f55a6-c743-4225-b566-e03655…"
0,3115,"""nodes/mother/lab/weyland-platf…","""upsert""",2,null,"""+ a projects-list.json that po…","""<vector dim=768>""","""bf9f5407edf559a218732a2f46ae31…","""feast-ui.yaml""","""code""","""554f55a6-c743-4225-b566-e03655…"
0,3116,"""nodes/mother/lab/weyland-platf…","""upsert""",3,null,""": replicas: 1 selector: …","""<vector dim=768>""","""bf9f5407edf559a218732a2f46ae31…","""feast-ui.yaml""","""code""","""554f55a6-c743-4225-b566-e03655…"


## 4 · Offsets and consumer groups — where reading stands

An offset is just a position in a partition's log, and a consumer group's **committed** offset is
how far *that group* has read. The two watermarks that bracket every partition are:

- **low (earliest)** — the oldest offset still retained (older ones aged out by retention).
- **high (log-end)** — one past the newest record; the next offset a producer will write.

`high - low` is how many records are currently readable — and for a group, `high - committed` is
its **lag** (how far behind live it is). Our reader used a *fresh* group and **never committed**,
so it has no committed offset at all: a brand-new `earliest` group effectively starts at `low` and
would only accrue a committed position if it committed as it read. That is exactly why our
read-only pass is invisible to every other consumer — we added no committed offset for anyone to
see.

Below we read the live watermarks per partition for the topic we consumed. The RAG indexer's own
group *does* commit, advancing through this same log independently of our throwaway group — that
independence between groups reading one log is the fan-out at the heart of the streaming model.

In [9]:
wm = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": f"nb80-watermarks-{uuid.uuid4()}",
               "enable.auto.commit": False})
rows = []
for p in sorted(pp for pp in md.topics[consume_topic].partitions):
    lo, hi = wm.get_watermark_offsets(TopicPartition(consume_topic, p), timeout=10)
    rows.append([p, lo, hi, hi - lo])
wm.close()

offsets_df = frame(rows, ["partition", "low (earliest)", "high (log-end)", "retained"])
total = sum(r[3] for r in rows)
print(f"{consume_topic}: {len(rows)} partitions, {total} records retained across the log")
offsets_df

rag.chunks: 3 partitions, 2755 records retained across the log


partition,low (earliest),high (log-end),retained
i64,i64,i64,i64
0,3108,4019,911
1,2523,3268,745
2,3274,4373,1099


## When to reach for streaming — and where this notebook sits

**Reach for the event stream when the question is "what *happened*?" rather than "what is the
state *now*?"** — when records must move between systems continuously, in order, and decoupled
from whoever consumes them.

| reach for streaming when… | why the log wins |
|---|---|
| **change-data-capture** — mirror every insert/update/delete out of a database | the log *is* the ordered change history; downstream stores fold it back into state |
| **real-time pipelines** — index, score, or roll up events as they arrive | consumers read from the head continuously; no batch window to wait for |
| **decoupling producers from consumers** | the producer appends and moves on; N independent groups read the same log at their own pace |
| **replay / rebuild** | offsets are just positions — a new consumer group replays history from `earliest` to rebuild a store from scratch |

**Reach for batch instead** when you want the *current* state and a periodic scan is enough: a
`SELECT` over a table (notebooks `20`–`22`), a dbt run over the lakehouse (`40`), a nightly
rollup. The log and the table are two views of the same data — *events in motion* versus *state
at rest* — and a healthy mesh uses both: stream the changes, materialize the state.

**Where this notebook fits in the streaming wave.** This one is the **backbone** — the Kafka API
and the Schema Registry that everything else plugs into:

- **Notebook `80` — this one.** Redpanda itself: connect, list topics, read a schema from the
  Registry, consume a bounded Avro batch. The transport layer, read-only.
- **Notebook `81` — Debezium CDC.** The `cdc.musicbrainz.public.cdc_demo` topic you saw in the
  listing here — a database's row-level changes captured *as* a stream, the canonical CDC use case.
- **Notebook `03` — Avro, the format.** The wire format underneath every message here, dissected
  offline: unions, defaults, schema evolution, and why row-oriented Avro is the streaming
  workhorse while columnar formats own the analytical tier.

> **Redpanda carries the events (this notebook); Debezium turns a database into one of those
> event streams (`81`); Avro is the shape each event takes on the wire (`03`).** Together they
> are the mesh's streaming layer — the log beneath the tables.